# Exercise 60 — Joins (`pd.merge`)

## Concept

`pd.merge` is pandas' SQL `JOIN`. Four join types via the `how=` parameter:

```python
pd.merge(left, right, on="user_id", how="inner")   # default — only matching keys
pd.merge(left, right, on="user_id", how="left")    # all left rows + matches
pd.merge(left, right, on="user_id", how="right")   # all right rows + matches
pd.merge(left, right, on="user_id", how="outer")   # union of keys, NaN where missing
```

### Different join column names

```python
pd.merge(left, right, left_on="customer_id", right_on="id", how="left")
```

### Composite keys

```python
pd.merge(left, right, on=["date", "region"], how="inner")
```

### Column name collisions

If both sides have a `name` column not in `on=`, pandas suffixes them — defaults to `_x` and `_y`. Override with `suffixes=("_left", "_right")`.

### `indicator=True` — debug aid

Adds a `_merge` column with values `"left_only"`, `"right_only"`, or `"both"`. Hugely useful when you suspect a join is dropping or duplicating rows.

```python
pd.merge(left, right, on="id", how="outer", indicator=True)
```

### Sanity check after every join

A join that silently duplicates rows or drops them is the most common DE bug. **Always check** `len(result)` against expectations and `result["_merge"].value_counts()` if you used `indicator=True`.

## Setup

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id":    [101, 102, 103, 104, 105],
    "customer_id": [1,   2,   1,   99,  3],     # 99 doesn't exist in customers
    "amount":      [10.0, 25.0, 7.5, 100.0, 12.0],
})

customers = pd.DataFrame({
    "id":     [1, 2, 3, 4],
    "name":   ["Alice", "Bob", "Carol", "Diana"],   # Diana has no orders
    "city":   ["Curitiba", "Rio", "SP", "Recife"],
})
print("orders:\n", orders)
print("customers:\n", customers)


## Task 1 — Inner join

Return an inner join of `orders` on `customers`, attaching each order's customer `name` and `city`. Match `orders.customer_id` to `customers.id`.

The result has columns `[order_id, customer_id, amount, name, city]` (in any order), and only orders with a matching customer.

```python
def inner_with_customer(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected: 4 rows (order_id 104 is dropped — customer_id 99 has no match).

In [ ]:
import pandas as pd

def inner_with_customer(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = inner_with_customer(orders, customers)
assert len(result) == 4
assert set(result["order_id"]) == {101, 102, 103, 105}
assert {"name", "city"} <= set(result.columns)
print("ok")


## Task 2 — Left join (keep all orders)

Return a left join: every order is preserved. Orders with no matching customer get `NaN` in `name` and `city`.

```python
def left_with_customer(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected: 5 rows. The row with order_id 104 has `NaN` for `name`.

In [ ]:
import pandas as pd

def left_with_customer(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = left_with_customer(orders, customers)
assert len(result) == 5
row_104 = result[result["order_id"] == 104].iloc[0]
assert pd.isna(row_104["name"])
assert pd.isna(row_104["city"])
print("ok")


## Task 3 — Outer join with indicator

Return an **outer** join with `indicator=True`. Then build a small reconciliation summary as a dict mapping `_merge` category → count.

```python
def reconcile_orders_customers(orders: pd.DataFrame, customers: pd.DataFrame) -> dict[str, int]:
    ...
```

Expected:
- `"both"`: 4 (orders that matched a customer)
- `"left_only"`: 1 (order 104, orphan)
- `"right_only"`: 1 (Diana, no orders)

This is exactly how you'd answer "are there orphan rows on either side?" in production.

In [ ]:
import pandas as pd

def reconcile_orders_customers(orders: pd.DataFrame, customers: pd.DataFrame) -> dict[str, int]:
    pass  # TODO

assert reconcile_orders_customers(orders, customers) == {
    "both": 4,
    "left_only": 1,
    "right_only": 1,
}
print("ok")


## Task 4 — Watch for row explosion

Duplicate keys can multiply rows. The `bonuses` table has the same customer twice:

```python
bonuses = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "bonus":       [10, 5, 20],
})
```

Return an inner join of `orders` and `bonuses` on `customer_id`, and tell us how many rows the result has. With customer 1 appearing twice in `bonuses` and twice in `orders`, the join produces **2 × 2 = 4** rows just for customer 1.

```python
def join_with_bonuses(orders: pd.DataFrame, bonuses: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected: 5 rows total (4 for customer 1 + 1 for customer 2).

In production, run `.duplicated(subset=key).sum()` on each side before joining if you're not sure the key is unique. A surprise multiplication can quietly inflate downstream metrics.

In [ ]:
import pandas as pd

bonuses = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "bonus":       [10, 5, 20],
})

def join_with_bonuses(orders: pd.DataFrame, bonuses: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = join_with_bonuses(orders, bonuses)
assert len(result) == 5
assert (result["customer_id"] == 1).sum() == 4   # row explosion for customer 1
print("ok")
